# Polynomial Regression

This notebook is one standalone implementation for the ML capstone Review 1. Run cells top-to-bottom. All notebooks use `random_state=42` and the same 80:20 split for fair comparison.

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")


## 2. Dataset audit and preprocessing

In [ ]:
# Dataset and target
DATA_URL = "https://raw.githubusercontent.com/rohithtej2109/Smart-car-deals/b438f32fc37ae91e6f8a72ef912705b2c48b427f/cardekho_dataset.csv"
TARGET = "selling_price"

df = pd.read_csv(DATA_URL)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Keep the target numeric and remove exact duplicate rows.
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.drop_duplicates().dropna(subset=[TARGET]).copy()

# Feature engineering: car_age captures depreciation better than raw year alone.
if "year" in df.columns:
    df["car_age"] = pd.Timestamp.now().year - pd.to_numeric(df["year"], errors="coerce")
    df["car_age"] = df["car_age"].clip(lower=0)

# Avoid using a high-cardinality name column as a raw categorical feature.
if "name" in df.columns:
    df = df.drop(columns=["name"])

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Same split for every regression notebook.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features)
])

print("Shape:", df.shape)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 3. Required EDA

In [ ]:
display(df.head())
display(df.describe(include="all").T)
print("Missing values:\n", df.isna().sum())
plt.figure(figsize=(8,4)); sns.histplot(y, kde=True); plt.title("Target distribution: selling_price"); plt.xlabel("Selling price"); plt.tight_layout(); plt.show()
plt.figure(figsize=(10,6)); sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="coolwarm", center=0); plt.title("Numeric correlation heatmap"); plt.tight_layout(); plt.show()
for col in [c for c in ["year","km_driven","car_age"] if c in df.columns]:
    plt.figure(figsize=(7,4)); sns.scatterplot(data=df, x=col, y=TARGET, alpha=.5); plt.title(f"{col} vs selling_price"); plt.tight_layout(); plt.show()


## 4. Model training and optional tuning

In [ ]:
base_model = Pipeline([('poly', PolynomialFeatures(include_bias=False)), ('reg', LinearRegression())])
model = Pipeline([("preprocessor", preprocessor), ("model", base_model)])


In [ ]:
param_grid = {'model__poly__degree':[2,3]}
search = GridSearchCV(model, param_grid, cv=5, scoring="r2", n_jobs=-1)
search.fit(X_train, y_train)
model = search.best_estimator_
print("Best parameters:", search.best_params_)


## 5. Evaluation

In [ ]:
pred = model.predict(X_test)
results = pd.DataFrame([{
    "Model": 'Polynomial Regression',
    "R2": r2_score(y_test, pred),
    "RMSE": mean_squared_error(y_test, pred, squared=False),
    "MAE": mean_absolute_error(y_test, pred)
}])
display(results)
cv_r2 = cross_val_score(model, X, y, cv=5, scoring="r2", n_jobs=-1)
print("5-fold CV R2 mean:", cv_r2.mean())
print("5-fold CV R2 std:", cv_r2.std())


## 6. Required visualisations / interpretation

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(y_test, pred, alpha=.6)
lims=[min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, "r--", label="Ideal prediction")
ax.set_xlabel("Actual selling price"); ax.set_ylabel("Predicted selling price")
ax.set_title("Predicted vs Actual"); ax.legend(); plt.tight_layout(); plt.show()

residuals = y_test - pred
plt.figure(figsize=(8,4)); sns.scatterplot(x=pred, y=residuals, alpha=.6)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted selling price"); plt.ylabel("Residual")
plt.title("Residual plot"); plt.tight_layout(); plt.show()

if hasattr(model.named_steps["model"], "feature_importances_"):
    names = model.named_steps["preprocessor"].get_feature_names_out()
    imp = pd.Series(model.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False).head(15)
    plt.figure(figsize=(8,5)); imp.sort_values().plot(kind="barh")
    plt.title("Top feature importances"); plt.xlabel("Importance"); plt.tight_layout(); plt.show()


**Interpretation note:** Discuss whether the model underfits/overfits, the meaning of R²/RMSE/MAE, and which features appear influential. Do not copy generic observations; write observations based on your actual output.